In [2]:
from pathlib import Path
import sys
import importlib

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src/config.py").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Nao encontrei src/config.py. Abra o notebook dentro do projeto.")
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import config
config = importlib.reload(config)

import pandas as pd

config.entrar_na_raiz()
base_cafe = config.carregar_base_cafe()

base_cafe.head()


,Código,Descrição,Departamento,Categoria/Setor,Marca,Famlíia,Código do Fabricante,Código de Barras,SKU Monitorado,Website Monitorado,URL Produto Monitorado,Data,Média Preço Normal,Média Preço Oferta
0,1038723,Café Torrado e Moído Extraforte Pilão Pacote 500g,Alimentos,Básico da despensa,pa,NaN,NaN,7896089013399,1038723,www.paodeacucar.com,https://www.paodeacucar.com/produto/292186/caf...,2026-03-08,27.49,27.49
1,1038723,Café Torrado e Moído Extraforte Pilão Pacote 500g,Alimentos,Básico da despensa,pa,NaN,NaN,7896089013399,1038723,www.paodeacucar.com,https://www.paodeacucar.com/produto/292186/caf...,2026-03-07,27.49,27.49
2,1038723,Café Torrado e Moído Extraforte Pilão Pacote 500g,Alimentos,Básico da despensa,pa,NaN,NaN,7896089013399,1038723,www.paodeacucar.com,https://www.paodeacucar.com/produto/292186/caf...,2026-03-06,27.49,27.49
3,183567,Café Torrado e Moído Extraforte 3 Corações Pac...,Alimentos,Básico da despensa,pa,NaN,NaN,7896005801529,183567,www.paodeacucar.com,https://www.paodeacucar.com/produto/57156/cafe...,2026-03-08,27.99,27.99
4,183567,Café Torrado e Moído Extraforte 3 Corações Pac...,Alimentos,Básico da despensa,pa,NaN,NaN,7896005801529,183567,www.paodeacucar.com,https://www.paodeacucar.com/produto/57156/cafe...,2026-03-07,27.99,27.99


In [3]:
base_cafe.info()

<class 'pandas.DataFrame'>
RangeIndex: 1047 entries, 0 to 1046
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Código                  1047 non-null   int64         
 1   Descrição               1047 non-null   str           
 2   Departamento            429 non-null    str           
 3   Categoria/Setor         429 non-null    str           
 4   Marca                   1047 non-null   str           
 5   Famlíia                 0 non-null      float64       
 6   Código do Fabricante    0 non-null      float64       
 7   Código de Barras        1047 non-null   int64         
 8   SKU Monitorado          1047 non-null   int64         
 9   Website Monitorado      1047 non-null   str           
 10  URL Produto Monitorado  1047 non-null   str           
 11  Data                    1047 non-null   datetime64[us]
 12  Média Preço Normal      908 non-null    float64       
 13 

In [4]:
base_limpa = base_cafe.drop(columns=[
    "Famlíia",
    "Código do Fabricante",
    "URL Monitor",
    "Desvio Média Preço %"
], errors="ignore")

In [5]:
base_limpa = base_limpa.dropna(subset=["Média Preço Normal"])

In [6]:
base_limpa["Website Monitorado"].value_counts()

Website Monitorado
www.paodeacucar.com    908
Name: count, dtype: int64

In [7]:
base_limpa["Código"].nunique()

336

In [8]:
base_recente = (
    base_limpa
    .sort_values("Data")
    .groupby("Código")
    .last()
    .reset_index()
)

In [9]:
base_recente.head()

,Código,Descrição,Departamento,Categoria/Setor,Marca,Código de Barras,SKU Monitorado,Website Monitorado,URL Produto Monitorado,Data,Média Preço Normal,Média Preço Oferta
0,106498,Café Torrado e Moído Tradicional Café Brasilei...,Alimentos,Básico da despensa,Café Brasileiro,7891018001386,106498,www.paodeacucar.com,https://www.paodeacucar.com/produto/62071/cafe...,2026-03-08,7.79,7.79
1,183567,Café Torrado e Moído Extraforte 3 Corações Pac...,Alimentos,Básico da despensa,pa,7896005801529,183567,www.paodeacucar.com,https://www.paodeacucar.com/produto/57156/cafe...,2026-03-08,27.99,27.99
2,250665,Café Torrado e Moído Tradicional Caboclo Pacot...,Alimentos,Básico da despensa,Caboclo,7896089011470,250665,www.paodeacucar.com,https://www.paodeacucar.com/produto/152018/caf...,2026-03-08,34.99,34.99
3,250672,Café Solúvel Granulado Extraforte Nescafé Orig...,Alimentos,Básico da despensa,pa,7891000300503,250672,www.paodeacucar.com,https://www.paodeacucar.com/produto/152381/caf...,2026-03-08,27.19,27.19
4,250702,Café Torrado e Moído Tradicional Pilão Pacote ...,NaN,NaN,Pilão,7896089011982,250702,www.paodeacucar.com,https://www.paodeacucar.com/produto/152472/caf...,2026-03-08,27.99,27.99


In [10]:
base_cafe.to_csv(PROJECT_ROOT / "base/base_cafe.csv", index=False)